<a href="https://colab.research.google.com/github/Eliekh2/MSBA-Projects/blob/main/Nigerian_E_wallet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Data Ingestion

The goal is to load the Nigerian Banking Mobile Dataset from HuggingFace into Colab.

The file format is parquet, this is useful for analytics.


###Installing HuggingFace

In [ ]:
pip install pandas pyarrow huggingface_hub

In [ ]:
pip install fsspec

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

raw_mobile_money.to_parquet(
    "/content/drive/MyDrive/raw_mobile_money.parquet",
    index=False
)

In [ ]:
DATA_PATH = "/content/drive/MyDrive/Customer Churn E-Wallets/"

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import pandas as pd

raw_mobile_money = pd.read_parquet("hf://datasets/electricsheepafrica/nigerian-banking-mobile-money/nigerian_mobile_money_full.parquet")

###Ingestion Validation

Objective:
Verify that the dataset loaded correctly and assess its structure.

Checks performed:

- shape (rows, columns)
- column names
- preview of data
- data types
- missing values
- duplicates

In [ ]:
print(raw_mobile_money.shape)
print(raw_mobile_money.columns.tolist())
raw_mobile_money.head()

In [ ]:
print(raw_mobile_money.info())
print(raw_mobile_money.isnull().sum())
print(raw_mobile_money.duplicated().sum())

###Raw Layer Storage

Objective:
Persist the ingested dataset in its original form to create a raw data layer.

Why:

ensures reproducibility
separates ingestion from transformation
preserves original data

In [ ]:
raw_mobile_money.to_parquet(
    "/content/raw_mobile_money.parquet",
    index=False
)

###Staging Layer Creation

Objective:
Create a cleaned version of the dataset for downstream processing.

In [ ]:
stg_mobile_money = raw_mobile_money.copy()

###Data Cleaning & Transformation
Convert Timestamp

Goal: Ensure proper datetime format for time-based analysis

In [ ]:
stg_mobile_money["timestamp"] = pd.to_datetime(
    stg_mobile_money["timestamp"],
    errors="coerce"
)

###Rename Columns

Goal: Standardize naming for consistency across the pipeline

In [ ]:
stg_mobile_money = stg_mobile_money.rename(columns={
    "timestamp": "txn_timestamp"
})

###Remove Duplicates

Goal: Ensure uniqueness of transactions

In [ ]:
stg_mobile_money = stg_mobile_money.drop_duplicates(
    subset=["transaction_id"]
)

###Handle Missing Values

Goal: Remove critical null records

In [ ]:
stg_mobile_money = stg_mobile_money.dropna(
    subset=["wallet_id", "amount_ngn"]
)

###Fix Data Types

Goal: Ensure numeric fields are properly typed

In [ ]:
stg_mobile_money["amount_ngn"] = pd.to_numeric(
    stg_mobile_money["amount_ngn"], errors="coerce"
)

stg_mobile_money["fee_ngn"] = pd.to_numeric(
    stg_mobile_money["fee_ngn"], errors="coerce"
)

###Staging Layer Storage

Objective:
Save the cleaned dataset for downstream analytics and modeling.

In [ ]:
import os

os.makedirs(DATA_PATH + "staging", exist_ok=True)

In [ ]:
stg_mobile_money.to_parquet(
    DATA_PATH + "staging/stg_mobile_money.parquet",
    index=False
)